# Exp 32: EfficientNet-B0, 5 folds

Card: [exp32_efficientnet_b0_5fold.md](../../docs/neuroslope/experiments/exp32_efficientnet_b0_5fold.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from helpers.config import ROOT
from helpers.config import VIP
from helpers.cv import CVResult
from helpers.gradcam import GradCamFigures
from helpers.plots import pr_curves, save_figures, use_style
from core.metrics import best_f1
from helpers.config import CLASSES

N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.yaml"
EXP26 = CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp32_efficientnet_b0_5fold"

# dataviz reference palette, light mode
SERIES = {"exp26": "#eb6834", "EfficientNet-B0": "#2a78d6"}
use_style()

## Training


In [ ]:
results = {
    "exp26": CVResult.load(EXP26),
    "EfficientNet-B0": CVResult.load(CONFIG),
}

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES, title="Out-of-fold rows of folds 0–4")
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Thresholds

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
thresholds = pd.concat({name: best_f1(r.labels, r.probs, CLASSES)[["threshold", "f1"]] for name, r in results.items()}, axis=1)
thresholds.round(3)

## Grad-CAM VIP: unseen images


In [ ]:
cam = GradCamFigures(CONFIG)
for cls in VIP:
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_validation.png"] = cam.positives(
        cls, n=N_POSITIVES
    )
plt.show()

In [ ]:
save_figures(figures, FIGURES)